# 03 Tone Modeling

Exploratory Cantonese tone and colloquial-error analysis using the Whisper API baseline predictions from `02_asr_baseline.ipynb`.

This notebook does not train a model or call an API. It reads prediction CSVs, analyzes Cantonese colloquial markers, detects written-Chinese replacement patterns, and prepares Jyutping/tone summaries for later tone-aware modeling.


## Setup


In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

def find_project_root():
    cwd = Path.cwd().resolve()
    env_root = os.getenv("PROJECT_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root))
    candidates += [cwd, *cwd.parents]
    candidates += [
        Path(r"D:/GitHub/Cantonese-Speech-AI"),
        Path(r"D:\GitHub\Cantonese-Speech-AI"),
        Path("/mnt/d/GitHub/Cantonese-Speech-AI"),
        Path("/content/Cantonese-Speech-AI"),
        Path("/content/drive/MyDrive/Cantonese-Speech-AI"),
    ]
    for candidate in candidates:
        if (candidate / "src" / "cantonese_speech_ai").exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Cannot find project root from cwd={cwd}")

ROOT = find_project_root()
os.environ["PROJECT_ROOT"] = str(ROOT)
load_dotenv(ROOT / ".env")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd

from cantonese_speech_ai.tone import (
    COLLOQUIAL_MARKERS,
    count_colloquial_markers,
    flag_written_replacements,
    jyutping_summary,
)

ROOT


WindowsPath('D:/GitHub/Cantonese-Speech-AI')

## Load Whisper baseline predictions


In [2]:
preferred = ROOT / "outputs" / "predictions" / "whisper_api_dev_20.csv"
prediction_files = sorted(
    (ROOT / "outputs" / "predictions").glob("whisper_api_dev_*.csv"),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

prediction_path = preferred if preferred.exists() else prediction_files[0]
predictions = pd.read_csv(prediction_path, encoding="utf-8-sig")

{
    "prediction_path": str(prediction_path),
    "rows": len(predictions),
    "mean_cer": predictions["cer"].mean(),
    "mean_wer": predictions["wer"].mean(),
}


{'prediction_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\predictions\\whisper_api_dev_20.csv',
 'rows': 20,
 'mean_cer': np.float64(0.3067809886192239),
 'mean_wer': np.float64(0.8833333333333334)}

In [3]:
display_columns = ["path", "sentence", "prediction", "cer", "wer"]
predictions.sort_values("cer", ascending=False)[display_columns].head(10)


,path,sentence,prediction,cer,wer
2,common_voice_yue_39013032.mp3,似我哋嗰度嘅,像我們那裡的,0.833333,1.0
9,common_voice_yue_39013286.mp3,仲係好窮嘅鄉下仔,還是很窮的鄉下人,0.625000,1.0
4,common_voice_yue_39013110.mp3,輸咗唔好喊，唔好走去搵阿媽求救,"輸了不要哭,不要去找媽媽求救。",0.600000,1.0
11,common_voice_yue_39013348.mp3,急都唔急嗰一陣,急都不急過一會兒,0.571429,1.0
17,common_voice_yue_39013850.mp3,我就係問緊先生住開邊個巿？,"我正在問先生,住開邊個市?",0.416667,2.0
15,common_voice_yue_39013938.mp3,做嘢做一半,工作做一半,0.400000,1.0
10,common_voice_yue_39013715.mp3,群主博覽羣書，博採眾長,"群主博覽群書,博彩頌祥。",0.363636,1.0
3,common_voice_yue_39013090.mp3,你從來都唔參加比賽，係乜嘢驅使你嚟呢度參賽？,"你從來都不參加比賽,是什麼驅使你來這裡參賽?",0.333333,1.0
6,common_voice_yue_39013129.mp3,都係手寫算數,還是手寫算數,0.333333,1.0
19,common_voice_yue_39013335.mp3,見到喇，唔該晒,"見到啦,唔該曬!",0.285714,1.0


## Cantonese colloquial marker coverage


In [4]:
marker_rows = []
for _, row in predictions.iterrows():
    ref_counts = count_colloquial_markers(str(row["normalized_sentence"]))
    pred_counts = count_colloquial_markers(str(row["normalized_prediction"]))
    for marker in COLLOQUIAL_MARKERS:
        marker_rows.append(
            {
                "path": row["path"],
                "marker": marker,
                "reference_count": ref_counts[marker],
                "prediction_count": pred_counts[marker],
                "lost_count": max(ref_counts[marker] - pred_counts[marker], 0),
                "cer": row["cer"],
            }
        )

marker_df = pd.DataFrame(marker_rows)
marker_summary = (
    marker_df.groupby("marker", as_index=False)[["reference_count", "prediction_count", "lost_count"]]
    .sum()
    .sort_values(["lost_count", "reference_count"], ascending=False)
)
marker_summary


,marker,reference_count,prediction_count,lost_count
5,唔,9,4,5
7,嘅,4,2,2
2,呢,3,1,2
6,嗰,2,0,2
0,乜嘢,2,1,1
3,咗,1,0,1
4,哋,1,0,1
8,嚟,1,0,1
9,畀,1,0,1
1,俾,0,1,0


## Written-Chinese replacement flags


In [5]:
analysis = predictions.copy()
analysis["written_replacement_flags"] = analysis.apply(
    lambda row: flag_written_replacements(
        str(row["normalized_sentence"]),
        str(row["normalized_prediction"]),
    ),
    axis=1,
)
analysis["written_replacement_count"] = analysis["written_replacement_flags"].map(len)
analysis["has_written_replacement"] = analysis["written_replacement_count"] > 0

analysis.loc[
    analysis["has_written_replacement"],
    ["path", "sentence", "prediction", "cer", "written_replacement_flags"],
].sort_values("cer", ascending=False).head(20)


,path,sentence,prediction,cer,written_replacement_flags
2,common_voice_yue_39013032.mp3,似我哋嗰度嘅,像我們那裡的,0.833333,"[嘅->的, 哋->們, 嗰->那]"
9,common_voice_yue_39013286.mp3,仲係好窮嘅鄉下仔,還是很窮的鄉下人,0.625000,[嘅->的]
4,common_voice_yue_39013110.mp3,輸咗唔好喊，唔好走去搵阿媽求救,"輸了不要哭,不要去找媽媽求救。",0.600000,"[唔->不, 咗->了]"
11,common_voice_yue_39013348.mp3,急都唔急嗰一陣,急都不急過一會兒,0.571429,[唔->不]
3,common_voice_yue_39013090.mp3,你從來都唔參加比賽，係乜嘢驅使你嚟呢度參賽？,"你從來都不參加比賽,是什麼驅使你來這裡參賽?",0.333333,"[唔->不, 嚟->來, 乜嘢->什麼, 呢->這]"
8,common_voice_yue_39013034.mp3,呢個真係超搞笑啊！,這個真是超搞笑啊!,0.250000,[呢->這]
12,common_voice_yue_39013350.mp3,我敢寫包單，你今次唔會作嘔或者暈低,"我敢寫包單,你今次不會作嘔或暈倒。",0.176471,[唔->不]


## Jyutping and tone summaries


In [6]:
def add_jyutping_columns(frame):
    frame = frame.copy()
    ref_summary = frame["normalized_sentence"].map(lambda text: jyutping_summary(str(text)))
    pred_summary = frame["normalized_prediction"].map(lambda text: jyutping_summary(str(text)))
    frame["reference_jyutping"] = ref_summary.map(lambda item: item["jyutping"])
    frame["prediction_jyutping"] = pred_summary.map(lambda item: item["jyutping"])
    frame["reference_tones"] = ref_summary.map(lambda item: " ".join(item["tones"]))
    frame["prediction_tones"] = pred_summary.map(lambda item: " ".join(item["tones"]))
    frame["reference_tone_count"] = ref_summary.map(lambda item: len(item["tones"]))
    frame["prediction_tone_count"] = pred_summary.map(lambda item: len(item["tones"]))
    frame["tone_count_delta"] = frame["reference_tone_count"] - frame["prediction_tone_count"]
    return frame

analysis = add_jyutping_columns(analysis)
analysis[
    [
        "path",
        "normalized_sentence",
        "normalized_prediction",
        "reference_jyutping",
        "prediction_jyutping",
        "tone_count_delta",
        "cer",
    ]
].sort_values("cer", ascending=False).head(10)


,path,normalized_sentence,normalized_prediction,reference_jyutping,prediction_jyutping,tone_count_delta,cer
2,common_voice_yue_39013032.mp3,似我哋嗰度嘅,像我們那裡的,ci5 ngo5 dei6 go2 dou6 ge3,zoeng6 ngo5 mun4 naa5 leoi5 dik1,0,0.833333
9,common_voice_yue_39013286.mp3,仲係好窮嘅鄉下仔,還是很窮的鄉下人,zung6 hai6 hou2 kung4 ge3 hoeng1 haa2 zai2,waan4 si6 han2 kung4 dik1 hoeng1 haa2 jan4,0,0.625000
4,common_voice_yue_39013110.mp3,輸咗唔好喊 唔好走去搵阿媽求救,輸了不要哭 不要去找媽媽求救,syu1 zo2 m4 hou2 haam3 m4 hou2 zau2 heoi3 wan2...,syu1 liu5 bat1 jiu3 huk1 bat1 jiu3 heoi3 zaau2...,1,0.600000
11,common_voice_yue_39013348.mp3,急都唔急嗰一陣,急都不急過一會兒,gap1 dou1 m4 gap1 go2 jat1 zan6,gap1 dou1 bat1 gap1 gwo3 jat1 wui6 ji4,-1,0.571429
17,common_voice_yue_39013850.mp3,我就係問緊先生住開邊個巿,我正在問先生 住開邊個市,ngo5 zau6 hai6 man6 gan2 sin1 saang1 zyu6 hoi1...,ngo5 zing3 zoi6 man6 sin1 saang1 zyu6 hoi1 bin...,1,0.416667
15,common_voice_yue_39013938.mp3,做嘢做一半,工作做一半,zou6 je5 zou6 jat1 bun3,gung1 zok3 zou6 jat1 bun3,0,0.400000
10,common_voice_yue_39013715.mp3,群主博覽羣書 博採眾長,群主博覽群書 博彩頌祥,kwan4 zyu2 bok3 laam5 kwan4 syu1 bok3 coi2 zun...,kwan4 zyu2 bok3 laam5 kwan4 syu1 bok3 coi2 zun...,0,0.363636
3,common_voice_yue_39013090.mp3,你從來都唔參加比賽 係乜嘢驅使你嚟呢度參賽,你從來都不參加比賽 是什麼驅使你來這裡參賽,nei5 cung4 loi4 dou1 m4 caam1 gaa1 bei2 coi3 h...,nei5 cung4 loi4 dou1 bat1 caam1 gaa1 bei2 coi3...,0,0.333333
6,common_voice_yue_39013129.mp3,都係手寫算數,還是手寫算數,dou1 hai6 sau2 se2 syun3 sou3,waan4 si6 sau2 se2 syun3 sou3,0,0.333333
19,common_voice_yue_39013335.mp3,見到喇 唔該晒,見到啦 唔該曬,gin3 dou2 laa3 m4 goi1 saai3,gin3 dou2 laa1 m4 goi1 saai3,0,0.285714


## Error taxonomy output


In [7]:
taxonomy = analysis.copy()
taxonomy["error_category"] = "low_error"
taxonomy.loc[taxonomy["cer"] >= 0.25, "error_category"] = "high_cer"
taxonomy.loc[taxonomy["has_written_replacement"], "error_category"] = "written_style_replacement"
taxonomy.loc[
    (taxonomy["cer"] >= 0.25) & taxonomy["has_written_replacement"],
    "error_category",
] = "high_cer_written_style"
taxonomy.loc[
    (taxonomy["tone_count_delta"].abs() >= 3) & (taxonomy["cer"] >= 0.25),
    "error_category",
] = "tone_sequence_changed"

taxonomy_columns = [
    "path",
    "sentence",
    "prediction",
    "cer",
    "wer",
    "error_category",
    "written_replacement_flags",
    "reference_jyutping",
    "prediction_jyutping",
    "reference_tones",
    "prediction_tones",
    "tone_count_delta",
]

taxonomy_output = taxonomy[taxonomy_columns].sort_values("cer", ascending=False)
analysis_path = ROOT / "outputs" / "analysis" / "tone_error_taxonomy_dev_20.csv"
analysis_path.parent.mkdir(parents=True, exist_ok=True)
taxonomy_output.to_csv(analysis_path, index=False, encoding="utf-8-sig")

{
    "analysis_path": str(analysis_path),
    "rows": len(taxonomy_output),
    "category_counts": taxonomy_output["error_category"].value_counts().to_dict(),
}


{'analysis_path': 'D:\\GitHub\\Cantonese-Speech-AI\\outputs\\analysis\\tone_error_taxonomy_dev_20.csv',
 'rows': 20,
 'category_counts': {'low_error': 7,
  'high_cer_written_style': 6,
  'high_cer': 6,
  'written_style_replacement': 1}}

In [8]:
taxonomy_output.head(20)


,path,sentence,prediction,cer,wer,error_category,written_replacement_flags,reference_jyutping,prediction_jyutping,reference_tones,prediction_tones,tone_count_delta
2,common_voice_yue_39013032.mp3,似我哋嗰度嘅,像我們那裡的,0.833333,1.000000,high_cer_written_style,"[嘅->的, 哋->們, 嗰->那]",ci5 ngo5 dei6 go2 dou6 ge3,zoeng6 ngo5 mun4 naa5 leoi5 dik1,5 5 6 2 6 3,6 5 4 5 5 1,0
9,common_voice_yue_39013286.mp3,仲係好窮嘅鄉下仔,還是很窮的鄉下人,0.625000,1.000000,high_cer_written_style,[嘅->的],zung6 hai6 hou2 kung4 ge3 hoeng1 haa2 zai2,waan4 si6 han2 kung4 dik1 hoeng1 haa2 jan4,6 6 2 4 3 1 2 2,4 6 2 4 1 1 2 4,0
4,common_voice_yue_39013110.mp3,輸咗唔好喊，唔好走去搵阿媽求救,"輸了不要哭,不要去找媽媽求救。",0.600000,1.000000,high_cer_written_style,"[唔->不, 咗->了]",syu1 zo2 m4 hou2 haam3 m4 hou2 zau2 heoi3 wan2...,syu1 liu5 bat1 jiu3 huk1 bat1 jiu3 heoi3 zaau2...,1 2 4 2 3 4 2 2 3 2 3 1 4 3,1 5 1 3 1 1 3 3 2 4 1 4 3,1
11,common_voice_yue_39013348.mp3,急都唔急嗰一陣,急都不急過一會兒,0.571429,1.000000,high_cer_written_style,[唔->不],gap1 dou1 m4 gap1 go2 jat1 zan6,gap1 dou1 bat1 gap1 gwo3 jat1 wui6 ji4,1 1 4 1 2 1 6,1 1 1 1 3 1 6 4,-1
17,common_voice_yue_39013850.mp3,我就係問緊先生住開邊個巿？,"我正在問先生,住開邊個市?",0.416667,2.000000,high_cer,[],ngo5 zau6 hai6 man6 gan2 sin1 saang1 zyu6 hoi1...,ngo5 zing3 zoi6 man6 sin1 saang1 zyu6 hoi1 bin...,5 6 6 6 2 1 1 6 1 1 3 1,5 3 6 6 1 1 6 1 1 3 5,1
15,common_voice_yue_39013938.mp3,做嘢做一半,工作做一半,0.400000,1.000000,high_cer,[],zou6 je5 zou6 jat1 bun3,gung1 zok3 zou6 jat1 bun3,6 5 6 1 3,1 3 6 1 3,0
10,common_voice_yue_39013715.mp3,群主博覽羣書，博採眾長,"群主博覽群書,博彩頌祥。",0.363636,1.000000,high_cer,[],kwan4 zyu2 bok3 laam5 kwan4 syu1 bok3 coi2 zun...,kwan4 zyu2 bok3 laam5 kwan4 syu1 bok3 coi2 zun...,4 2 3 5 4 1 3 2 3 6,4 2 3 5 4 1 3 2 6 4,0
3,common_voice_yue_39013090.mp3,你從來都唔參加比賽，係乜嘢驅使你嚟呢度參賽？,"你從來都不參加比賽,是什麼驅使你來這裡參賽?",0.333333,1.000000,high_cer_written_style,"[唔->不, 嚟->來, 乜嘢->什麼, 呢->這]",nei5 cung4 loi4 dou1 m4 caam1 gaa1 bei2 coi3 h...,nei5 cung4 loi4 dou1 bat1 caam1 gaa1 bei2 coi3...,5 4 4 1 4 1 1 2 3 6 1 5 1 2 5 4 1 6 1 3,5 4 4 1 1 1 1 2 3 6 6 1 1 2 5 4 2 5 1 3,0
6,common_voice_yue_39013129.mp3,都係手寫算數,還是手寫算數,0.333333,1.000000,high_cer,[],dou1 hai6 sau2 se2 syun3 sou3,waan4 si6 sau2 se2 syun3 sou3,1 6 2 2 3 3,4 6 2 2 3 3,0
19,common_voice_yue_39013335.mp3,見到喇，唔該晒,"見到啦,唔該曬!",0.285714,1.000000,high_cer,[],gin3 dou2 laa3 m4 goi1 saai3,gin3 dou2 laa1 m4 goi1 saai3,3 2 3 4 1 3,3 2 1 4 1 3,0


## Next modeling step

Use this taxonomy to decide which tone-aware experiments matter most. For the current Whisper API baseline, the most important first issue is colloquial Cantonese preservation, especially cases where the prediction is valid written Chinese but no longer matches the Common Voice Yue transcript.
